In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_clinica")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
# UDF para clasificar riesgo de deserción del paciente
def riesgo_desercion(estado):
    if estado == "Inasistencia":
        return "Alto"
    elif estado == "Cancelado":
        return "Medio"
    else:
        return "Bajo"

riesgo_udf = F.udf(riesgo_desercion, StringType())

In [0]:
df_citas = spark.table(f"{catalogo}.{esquema_source}.citas")
df_pacientes = spark.table(f"{catalogo}.{esquema_source}.pacientes")
df_medicos = spark.table(f"{catalogo}.{esquema_source}.medicos")
df_especialidades = spark.table(f"{catalogo}.{esquema_source}.especialidades")

In [0]:
df_citas_join = df_citas.alias("c")\
    .join(df_pacientes.alias("p"), col("c.id_paciente") == col("p.id_paciente"), "inner")\
    .join(df_medicos.alias("m"), col("c.id_medico") == col("m.id_medico"), "inner")\
    .join(df_especialidades.alias("e"), col("c.id_especialidad") == col("e.id_especialidad"), "inner")

In [0]:
df_citas_select = df_citas_join.select(
    col("c.id_cita"),
    col("c.fecha_programada"),
    year(col("c.fecha_programada")).alias("anio"),
    month(col("c.fecha_programada")).alias("mes"),
    date_format(col("c.fecha_programada"), "EEEE").alias("dia_semana"),
    hour(col("c.fecha_programada")).alias("hora"),
    col("m.turno"),
    col("p.id_paciente"),
    col("p.nombre_completo").alias("nombre_paciente"),
    col("p.sexo").alias("sexo_paciente"),
    col("p.edad").alias("edad_paciente"),
    col("p.distrito").alias("distrito_paciente"),
    col("p.tipo_seguro"),
    col("m.id_medico"),
    col("m.nombre_completo").alias("nombre_medico"),
    col("e.nombre_especialidad"),
    col("e.area").alias("area_especialidad"),
    col("c.estado"),
    col("c.motivo_cancelacion")
)

In [0]:
df_citas_riesgo = df_citas_select.withColumn("riesgo_desercion", riesgo_udf(col("estado")))

In [0]:
df_citas_riesgo.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.citas_detalle")

In [0]:
# Transform atenciones
df_atenciones = spark.table(f"{catalogo}.{esquema_source}.atenciones")
df_pagos = spark.table(f"{catalogo}.{esquema_source}.pagos")

In [0]:
df_atenciones_join = df_atenciones.alias("a")\
    .join(df_pacientes.alias("p"), col("a.id_paciente") == col("p.id_paciente"), "inner")\
    .join(df_medicos.alias("m"), col("a.id_medico") == col("m.id_medico"), "inner")\
    .join(df_especialidades.alias("e"), col("m.id_especialidad") == col("e.id_especialidad"), "inner")\
    .join(df_pagos.alias("pg"), col("a.id_atencion") == col("pg.id_atencion"), "inner")

In [0]:
df_atenciones_select = df_atenciones_join.select(
    col("a.id_atencion"),
    col("a.id_cita"),
    col("a.fecha_atencion"),
    col("p.id_paciente"),
    col("p.nombre_completo").alias("nombre_paciente"),
    col("p.distrito").alias("distrito_paciente"),
    col("p.tipo_seguro"),
    col("m.id_medico"),
    col("m.nombre_completo").alias("nombre_medico"),
    col("e.nombre_especialidad"),
    col("a.diagnostico"),
    col("a.tratamiento"),
    col("a.duracion_min"),
    col("a.requiere_seguimiento"),
    col("pg.monto_pagado"),
    col("pg.tipo_pago"),
    col("pg.estado_pago")
)

In [0]:
df_atenciones_select.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.atenciones_detalle")